<a href="https://colab.research.google.com/github/SaskTakeda/SaskTakeda/blob/main/trabalho3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests

# Definindo a URL, parâmetros e headers
url = "https://jsonplaceholder.typicode.com/posts"
params = {"userId": 2}
headers = {"User-Agent": "MeuProjetoAnaliseDados/1.0 (contato@email.com)"}

try:
    # 1.1, 1.2, 1.3 e 1.4: Fazendo a requisição com timeout
    resposta = requests.get(url, params=params, headers=headers, timeout=10)

    # Exibindo status code e URL final montada
    print(f"Código de Status HTTP: {resposta.status_code}")
    print(f"URL Final Montada: {resposta.url}")

    if resposta.status_code == 200:
        dados = resposta.json()
        print(f"\nTotal de registros retornados: {len(dados)}")
        print("Exemplo do primeiro resultado:", dados[0])

except requests.exceptions.RequestException as e:
    print(f"Ocorreu um erro na requisição: {e}")

Código de Status HTTP: 200
URL Final Montada: https://jsonplaceholder.typicode.com/posts?userId=2

Total de registros retornados: 10
Exemplo do primeiro resultado: {'userId': 2, 'id': 11, 'title': 'et ea vero quia laudantium autem', 'body': 'delectus reiciendis molestiae occaecati non minima eveniet qui voluptatibus\naccusamus in eum beatae sit\nvel qui neque voluptates ut commodi qui incidunt\nut animi commodi'}


In [2]:
import pandas as pd
import requests

ceps = ["01001000", "20040002", "30130010"]
dados_ceps = []

for cep in ceps:
    url_viacep = f"https://viacep.com.br/ws/{cep}/json/"
    try:
        resp = requests.get(url_viacep, timeout=5)
        if resp.status_code == 200:
            dados_ceps.append(resp.json())
    except requests.exceptions.RequestException as e:
        print(f"Erro ao consultar o CEP {cep}: {e}")

# Convertendo a lista de dicionários em um DataFrame do Pandas
df_ceps = pd.DataFrame(dados_ceps)
display(df_ceps)

,cep,logradouro,complemento,unidade,bairro,localidade,uf,estado,regiao,ibge,gia,ddd,siafi
0,01001-000,Praça da Sé,lado ímpar,,Sé,São Paulo,SP,São Paulo,Sudeste,3550308,1004,11,7107
1,20040-002,Avenida Rio Branco,de 128 a 144 - lado par,,Centro,Rio de Janeiro,RJ,Rio de Janeiro,Sudeste,3304557,,21,6001
2,30130-010,Praça Sete de Setembro,,,Centro,Belo Horizonte,MG,Minas Gerais,Sudeste,3106200,,31,4123


In [3]:
def baixar_imagem(url, nome_arquivo):
    try:
        resposta = requests.get(url, timeout=10)
        # Lança exceção caso haja erro HTTP (404, 500, etc.)
        resposta.raise_for_status()

        # Salvando o conteúdo bruto (binário) em formato .jpg
        with open(nome_arquivo, 'wb') as arquivo:
            arquivo.write(resposta.content)
        print(f"Sucesso! Imagem salva localmente como '{nome_arquivo}'.")

    except requests.exceptions.HTTPError as err:
        print(f"Erro HTTP na requisição: {err}")
    except requests.exceptions.RequestException as err:
        print(f"Erro de conexão: {err}")

# Executando o download da imagem aleatória do Picsum
url_imagem = "https://picsum.photos/400/400"
baixar_imagem(url_imagem, "imagem_aleatoria.jpg")

Sucesso! Imagem salva localmente como 'imagem_aleatoria.jpg'.


In [5]:
url_robots = "https://books.toscrape.com/robots.txt"
resp_robots = requests.get(url_robots, timeout=10)

print("--- Conteúdo do robots.txt ---")
print(resp_robots.text)

--- Conteúdo do robots.txt ---
<html>
<head><title>404 Not Found</title></head>
<body>
<center><h1>404 Not Found</h1></center>
<hr><center>nginx/1.21.6</center>
</body>
</html>



In [6]:
from bs4 import BeautifulSoup
import pandas as pd
import requests

url_site = "https://books.toscrape.com/"
resposta = requests.get(url_site, timeout=10)
resposta.encoding = 'utf-8'

# Analisando o HTML com BeautifulSoup
soup = BeautifulSoup(resposta.text, 'html.parser')

# Selecionando os 5 primeiros produtos
livros = soup.select('article.product_pod')[:5]

lista_livros = []
for livro in livros:
    titulo = livro.h3.a['title']
    preco = livro.select_one('.price_color').get_text()
    lista_livros.append({'Título': titulo, 'Preço': preco})

df_livros = pd.DataFrame(lista_livros)
display(df_livros)

# Salvando em arquivo CSV
df_livros.to_csv('livros_top5.csv', index=False, encoding='utf-8-sig')
print("\nDados salvos com sucesso em 'livros_top5.csv'.")

,Título,Preço
0,A Light in the Attic,£51.77
1,Tipping the Velvet,£53.74
2,Soumission,£50.10
3,Sharp Objects,£47.82
4,Sapiens: A Brief History of Humankind,£54.23



Dados salvos com sucesso em 'livros_top5.csv'.


In [7]:
import io
import pandas as pd
import requests

url_wiki = "https://pt.wikipedia.org/wiki/Lista_de_pa%C3%ADses_e_territ%C3%B3rios_por_popula%C3%A7%C3%A3o"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

resp_wiki = requests.get(url_wiki, headers=headers, timeout=10)

# Passando o conteúdo HTML em texto para o StringIO para que o read_html processe de forma segura
html_io = io.StringIO(resp_wiki.text)
tabelas = pd.read_html(html_io)

# Selecionando a primeira tabela da página
df_populacao = tabelas[0]
display(df_populacao.head(10))

,0,1
0,NaN,Wikicionário (dicionário livre)
1,NaN,Wikilivros (livros didáticos)
2,NaN,Wikiquote (citações)
3,NaN,Wikisource (biblioteca livre)
4,NaN,Wikiversidade (fontes de aprendizado livres)
5,NaN,Commons (imagens e media)
6,NaN,Wikinotícias (fonte de notícias livre)
7,NaN,Wikivoyage (guia de viagem)
8,NaN,Wikidata (base de dados)
9,NaN,Wikispecies (directório de espécies)
